<a href="https://colab.research.google.com/github/akatwrk/DeeplearningExp/blob/main/exp_4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
import zipfile
import os

# Mount your Google Drive
drive.mount('/content/drive')

# Exact path based on your Google Drive screenshot
zip_path = '/content/drive/MyDrive/deeplearning/Combined_Dataset.zip'
extract_path = '/content/dataset'

# Extract the zip file
with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

# Automatically locate 'train' and 'test' folders inside the extracted files
train_dir, test_dir = None, None
for root, dirs, _ in os.walk(extract_path):
    if 'train' in dirs and 'test' in dirs:
        train_dir = os.path.join(root, 'train')
        test_dir = os.path.join(root, 'test')
        break

print("Train directory found:", train_dir)
print("Test directory found:", test_dir)

Mounted at /content/drive
Train directory found: /content/dataset/Combined Dataset/train
Test directory found: /content/dataset/Combined Dataset/test


In [ ]:
import tensorflow as tf
from tensorflow.keras import mixed_precision

# Enable 16-bit mixed precision for 2x GPU speedup
mixed_precision.set_global_policy('mixed_float16')

BATCH_SIZE = 64
IMAGE_SIZE = (128, 128)

train_dataset = tf.keras.utils.image_dataset_from_directory(
    train_dir,
    image_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    label_mode='categorical',
    shuffle=True
)

test_dataset = tf.keras.utils.image_dataset_from_directory(
    test_dir,
    image_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    label_mode='categorical',
    shuffle=False
)

num_classes = len(train_dataset.class_names)

AUTOTUNE = tf.data.AUTOTUNE
train_dataset = train_dataset.cache().prefetch(buffer_size=AUTOTUNE)
test_dataset = test_dataset.cache().prefetch(buffer_size=AUTOTUNE)

Found 10240 files belonging to 4 classes.
Found 1279 files belonging to 4 classes.


In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Rescaling, GlobalAveragePooling2D, Lambda
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input

# --- Model 1: Lightweight Custom CNN ---
custom_model = Sequential([
    Rescaling(1./255, input_shape=(128, 128, 3)),
    Conv2D(32, 3, activation='relu'),
    MaxPooling2D(),
    Conv2D(64, 3, activation='relu'),
    MaxPooling2D(),
    Flatten(),
    Dense(64, activation='relu'),
    Dense(num_classes, activation='softmax', dtype='float32')
])
custom_model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

print("--> Training Custom CNN (3 Epochs)...")
custom_model.fit(train_dataset, validation_data=test_dataset, epochs=3)

# --- Model 2: Frozen MobileNetV2 Transfer Learning ---
base_model = MobileNetV2(weights='imagenet', include_top=False, input_shape=(128, 128, 3))
base_model.trainable = False

transfer_model = Sequential([
    Lambda(preprocess_input, input_shape=(128, 128, 3)),
    base_model,
    GlobalAveragePooling2D(),
    Dense(64, activation='relu'),
    Dense(num_classes, activation='softmax', dtype='float32')
])
transfer_model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

print("\n--> Training Transfer Learning Model (3 Epochs)...")
transfer_model.fit(train_dataset, validation_data=test_dataset, epochs=3)

/usr/local/lib/python3.13/dist-packages/keras/src/layers/preprocessing/data_layer.py:95: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


--> Training Custom CNN (3 Epochs)...
Epoch 1/3
160/160 ━━━━━━━━━━━━━━━━━━━━ 17s 58ms/step - accuracy: 0.6301 - loss: 0.8579 - val_accuracy: 0.6755 - val_loss: 0.7003
Epoch 2/3
160/160 ━━━━━━━━━━━━━━━━━━━━ 2s 15ms/step - accuracy: 0.8645 - loss: 0.3139 - val_accuracy: 0.8233 - val_loss: 0.4287
Epoch 3/3
160/160 ━━━━━━━━━━━━━━━━━━━━ 2s 15ms/step - accuracy: 0.9272 - loss: 0.1854 - val_accuracy: 0.8890 - val_loss: 0.2926
9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 2s 0us/step

--> Training Transfer Learning Model (3 Epochs)...
Epoch 1/3


/usr/local/lib/python3.13/dist-packages/keras/src/layers/core/lambda_layer.py:65: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


160/160 ━━━━━━━━━━━━━━━━━━━━ 79s 277ms/step - accuracy: 0.6801 - loss: 0.7356 - val_accuracy: 0.5927 - val_loss: 0.8557
Epoch 2/3
160/160 ━━━━━━━━━━━━━━━━━━━━ 2s 12ms/step - accuracy: 0.7840 - loss: 0.5041 - val_accuracy: 0.6185 - val_loss: 0.8206
Epoch 3/3
160/160 ━━━━━━━━━━━━━━━━━━━━ 2s 11ms/step - accuracy: 0.8121 - loss: 0.4453 - val_accuracy: 0.6474 - val_loss: 0.7758


In [ ]:
_, custom_acc = custom_model.evaluate(test_dataset, verbose=0)
_, transfer_acc = transfer_model.evaluate(test_dataset, verbose=0)

print("\n" + "=" * 42)
print(f"Custom CNN Accuracy:        {custom_acc * 100:.2f}%")
print(f"Transfer Learning Accuracy: {transfer_acc * 100:.2f}%")
print("=" * 42)


Custom CNN Accuracy:        88.90%
Transfer Learning Accuracy: 64.74%
